In [ ]:
!pip install "scikit-learn>=1.1" "POT>=0.9.3" numpy
!pip install tslearn

In [ ]:
import numpy as np
from tslearn.metrics import dtw as ts_dtw

def dtw_distance_series(x, y, sakoe_chiba_radius=None):
    """
    Tính khoảng cách DTW giữa hai chuỗi x, y.
    x, y: numpy array 1D hoặc 2D (đa biến, tslearn tự xử lý)
    sakoe_chiba_radius: nếu None thì DTW full, ngược lại dùng Sakoe-Chiba band.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    if sakoe_chiba_radius is None:
        dist = ts_dtw(x, y)
    else:
        dist = ts_dtw(
            x, y,
            global_constraint="sakoe_chiba",
            sakoe_chiba_radius=sakoe_chiba_radius
        )
    return float(dist)


In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def tcot_distance_series(x, y, lambda_pos: float = 1.0, reg: float = 0.1, num_iter: int = 1000):
    """
    Tính khoảng cách Temporally Coupled Optimal Transport (TCOT) giữa hai chuỗi x, y.

    Hỗ trợ:
      - CPU: nếu x, y là numpy.ndarray (hoặc list)  -> dùng NumPy + ot.sinkhorn
      - GPU: nếu x hoặc y là cupy.ndarray          -> dùng CuPy + ot.sinkhorn

    Parameters
    ----------
    x, y : array-like hoặc np.ndarray / cp.ndarray
        - 1D: shape (n,)
        - 2D: shape (n, d)

    lambda_pos : float
        Hệ số cho thành phần lệch thời gian trong hệ số ghép cặp.
        Với lambda_pos=1.0, công thức khớp: d_ij * (|i/n - j/m| + 1).

    reg : float
        Entropic regularization cho Sinkhorn.

    num_iter : int
        Số vòng lặp tối đa cho Sinkhorn.

    Returns
    -------
    float
        Giá trị OT(C) với ground cost TCOT.
    """
    # Chọn backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(x, (cp.ndarray,)) or isinstance(y, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # Đưa dữ liệu về đúng backend
    x = xp.asarray(x, dtype=xp.float64)
    y = xp.asarray(y, dtype=xp.float64)

    # Ép về dạng (n, d)
    if x.ndim == 1:
        x = x[:, None]
    if y.ndim == 1:
        y = y[:, None]

    n, dx = x.shape
    m, dy = y.shape
    if dx != dy:
        raise ValueError(f"Dimension mismatch: x dim={dx}, y dim={dy}")
    if n == 0 or m == 0:
        raise ValueError("Empty time series")

    # Cost đặc trưng: d_ij = ||x_i - y_j||^2
    diff = x[:, None, :] - y[None, :, :]   # (n, m, d)
    C_feat = xp.sum(diff * diff, axis=2)   # (n, m)

    # Theo công thức paper: i/n và j/m (chỉ số 1-based)
    t = xp.arange(1, n + 1, dtype=xp.float64) / float(n)
    s = xp.arange(1, m + 1, dtype=xp.float64) / float(m)

    # Theo ảnh: d_tilde(i,j) = d_ij * (|i/n - j/m| + 1)
    # Giữ API cũ bằng cách nhân thêm lambda_pos vào phần |.|.
    pos_diff = xp.abs(t[:, None] - s[None, :])  # (n, m)
    temporal_factor = 1.0 + lambda_pos * pos_diff

    # Ground cost TCOT (dạng nhân theo hệ số thời gian)
    C_base = C_feat * temporal_factor

    # Chuẩn hóa cost cho Sinkhorn (giống TAOT)
    med = xp.median(C_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    C = C_base / med

    # Trọng số đều
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    # Thêm stopThr để tăng tốc (giống TAOT)
    G = ot.sinkhorn(a, b, C, reg, numItermax=num_iter, stopThr=5e-3)
    cost = float(xp.sum(G * C_base))  # Dùng C_base (chưa chuẩn hóa) để tính distance

    return cost

In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def pow_distance(
    A,
    B,
    lam=10.0,          # giống TAOT: điều khiển reg = 1/lam cho Sinkhorn
    lam_order=1.0,     # hệ số cho regularization theo thứ tự (lambda_1)
    bandwidth=None,    # băng |i-j| cho phép match; nếu None sẽ auto
    tolerance=5e-3,
):
    """
    POW distance (phiên bản gần đúng theo tinh thần Partial Ordered Wasserstein),
    hỗ trợ GPU nếu A hoặc B là cupy.ndarray.

    Ý tưởng:
    - Ground cost: ||x_i - y_j||^2 + lam_order * |i/n - j/m|
    - Chỉ cho phép match trong một băng theo chỉ số |i - j| <= bandwidth
      (mô phỏng "partial" + hạn chế match xa).
    - Dùng Sinkhorn (entropic OT) như TAOT, reg = 1 / lam.

    Tham số:
      A, B      : (n,d), (m,d) NumPy hoặc CuPy array.
      lam       : tham số entropic (reg = 1/lam).
      lam_order : trọng số cho thành phần bảo toàn thứ tự.
      bandwidth : nếu None -> auto = 0.25 * max(n, m); nếu là số nguyên -> dùng trực tiếp.
      tolerance : ngưỡng dừng Sinkhorn.

    Trả về:
      distance_raw : scalar float (Python float).
    """
    # Chọn backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # Đưa dữ liệu về đúng backend
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    # Đảm bảo dạng (n, d)
    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # 1) Cost đặc trưng: ||x_i - y_j||^2
    diff = A[:, None, :] - B[None, :, :]
    M_base = xp.sum(diff * diff, axis=2)  # shape (n, m)

    # 2) Regularization tuyến tính theo thứ tự: lam_order * |i/n - j/m|
    #    (khác với TAOT: dùng |.|, không bình phương)
    posA = xp.linspace(1, n, n) / float(n)   # 1/n, 2/n, ..., 1
    posB = xp.linspace(1, m, m) / float(m)   # 1/m, ..., 1
    order_term = xp.abs(posA[:, None] - posB[None, :])

    M_base = M_base + lam_order * order_term

    # 3) Bandwidth (partial constraint): chỉ cho phép match trong |i - j| <= bandwidth
    #    Nếu không cho, ta đặt chi phí Sinkhorn rất lớn ở ngoài băng.
    if bandwidth is None:
        # auto: 1/4 độ dài lớn hơn, làm int >= 1
        max_len = max(n, m)
        bandwidth = max(1, int(0.25 * max_len))

    idx_i = xp.arange(n)[:, None]
    idx_j = xp.arange(m)[None, :]
    mask = (xp.abs(idx_i - idx_j) <= bandwidth)  # True nếu được phép match

    # 4) Chuẩn hoá cost cho Sinkhorn
    med = xp.median(M_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0

    C = M_base / med

    # Thêm penalty rất lớn ở ngoài băng (chỉ trong C dùng cho Sinkhorn)
    # để gần như cấm mass đi ra ngoài vùng cho phép.
    big_C = 1e3
    C = C + (~mask).astype(xp.float64) * big_C

    # 5) Khối lượng đều
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    # 6) Sinkhorn (POT tự nhận backend từ kiểu mảng)
    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # 7) Tính distance dùng cost "thật" M_base (không cộng big_C)
    distance_raw = float(xp.sum(P * M_base))
    return distance_raw


In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def taot_distance(A, B, lam=10.0, w=10.0, tolerance=5e-3):
    """
    TAOT distance, hỗ trợ GPU nếu A hoặc B là cupy.ndarray.

    - Nếu A/B là numpy array hoặc list -> chạy trên CPU (NumPy).
    - Nếu A hoặc B là cupy.ndarray -> convert cả hai sang CuPy và chạy Sinkhorn trên GPU.
    API (tham số/hàm trả về) giữ nguyên.
    """
    # Chọn backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # Đưa dữ liệu về đúng backend
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # Chuẩn hoá chỉ số thời gian t,s (z-score) bằng backend xp
    t = xp.linspace(1, n, n)
    s = xp.linspace(1, m, m)

    def _zscore(x):
        mu = x.mean()
        std = x.std()
        if float(std) == 0.0:
            return x * 0.0
        return (x - mu) / std

    t = _zscore(t)
    s = _zscore(s)

    # Ma trận cost M (data + term bảo toàn thứ tự)
    diff = A[:, None, :] - B[None, :, :]
    M = xp.sum(diff * diff, axis=2) + w * (t[:, None] - s[None, :]) ** 2

    # Chuẩn hoá để đưa về C cho Sinkhorn
    med = xp.median(M)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    C = M / med

    # Khối lượng đều
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    # POT sẽ tự nhận backend dựa trên kiểu mảng (NumPy hoặc CuPy)
    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # Tính distance trong cùng backend rồi ép về float Python
    distance_raw = float(xp.sum(P * M))
    return distance_raw


In [ ]:
import numpy as np
import ot

def polynomial(x, a, b):
    return a*x + b

def exponential(x, a, b, c):
    return a * np.exp(b*x + c)

def logarithm(x, a, b):
    return np.log(a*x + b)

def hyperbolic_tangent(x, a, b, c):
    return a * np.tanh(b*x + c)

def polynomial_with_degree(x, a, b, c):
    return a * pow(b*x, c)

def compute_f(function_info, x):
    match function_info[0]:
        case 'polynomial':
            return polynomial(x, function_info[1][0], function_info[1][1])
        case 'exponential':
            return exponential(x, function_info[1][0], function_info[1][1], function_info[1][2])
        case 'logarithm':
            return logarithm(x, function_info[1][0], function_info[1][1])
        case 'hyperbolic_tangent':
            return hyperbolic_tangent(x, function_info[1][0], function_info[1][1], function_info[1][2])
        case 'polynomial_with_degree':
            return polynomial_with_degree(x, function_info[1][0], function_info[1][1], function_info[1][2])

    return ValueError("Function not defined")

def compute_f_scale(function_info, x, i_scale, j_scale):
    '''Assume the input function is defined on the unit square.
    Scale the function according to the lengths of the two series.
    '''

    x_scaled = x / j_scale

    match function_info[0]:
        case 'polynomial':
            return i_scale * polynomial(x_scaled, function_info[1][0], function_info[1][1])
        case 'exponential':
            return i_scale * exponential(x_scaled, function_info[1][0], function_info[1][1], function_info[1][2])
        case 'logarithm':
            return i_scale * logarithm(x_scaled, function_info[1][0], function_info[1][1])
        case 'hyperbolic_tangent':
            return i_scale * hyperbolic_tangent(x_scaled, function_info[1][0], function_info[1][1], function_info[1][2])
        case 'polynomial_with_degree':
            return i_scale * polynomial_with_degree(x_scaled, function_info[1][0], function_info[1][1], function_info[1][2])
            
    return ValueError("Function not defined, valid function name:'polynomial', 'exponential', 'logarithm', 'hyperbolic_tangent', 'polynomial_with_degree'")

def compute_new_cost(old_D, alpha, F, LAMBDA3):
    '''
    Tính ma trận chi phí mới với vector trọng số mới
    (không scale)
    '''

    n = old_D.shape[0]
    m = old_D.shape[1]

    new_D = np.empty((n, m))

    for i in range(n):
        for j in range(m):
            new_D[i][j] = old_D[i][j] + LAMBDA3 * (i - float(np.dot(np.squeeze(np.asarray(alpha)), F[j])))**2 / (n**2)

    return new_D

def compute_new_cost2(old_D, w, F, LAMBDA1):
    '''Compute a new cost matrix using the new weight vector
    (takes series lengths into account).
    '''

    n = old_D.shape[0]
    m = old_D.shape[1]

    new_D = np.empty((n, m))

    for i in range(n):
        for j in range(m):
            new_D[i][j] = old_D[i][j] + LAMBDA1 * (i/n - float(np.dot(np.squeeze(np.asarray(w)), F[j]))/m)**2

    return new_D

def choose_initial_w(Y, V, num_function):
    '''Initialize the weight vector for the GOW objective.
    The vector has a single 1 and remaining entries equal to 0.
    '''

    initial_w = np.zeros((num_function, 1))
    min_S = np.Inf
    min_index = 0

    for i in range(num_function):
        sub = Y - V[:,[i]]
        sum_squared = np.sum(np.square(sub))

        if sum_squared < min_S:
            min_index = i
            min_S = sum_squared

    initial_w[min_index][0] = 1

    return initial_w

def gow_sinkhorn(a, b, D, function_list=[("polynomial", (1.0, 0)),], LAMBDA1=5, LAMBDA2=10, maxIter=15, epsilon=0.01, num_FW_iteration=100, show_details=False):
    '''Compute the GOW distance between two series.

    If the cost matrix D has shape n x m, the resulting transport matrix
    will also be n x m.

    Parameters
    ----------
    a : array-like, shape (dim_a,)
        Source histogram weights
    b : array-like, shape (dim_b,) or ndarray, shape (dim_b, n_hists)
        Target histogram weights
    D : array-like, shape (dim_a, dim_b)
        Loss/cost matrix
    function_list : list, optional
        Input functions used to control deformation paths
    LAMBDA1 : float, optional
        Regularization parameter for input functions
    LAMBDA2 : float, optional
        Regularization parameter for Sinkhorn
    maxIter: int, optional
        Maximum number of outer iterations (coordinate descent)
    epsilon: float, optional
        Stopping tolerance for weight updates
    num_FW_iteration: int, optional
        Number of Frank-Wolfe iterations
    show_details: bool, optional
        If True, return the GOW distance, transport matrix and weight vector

    Returns
    -------
    float
        GOW distance
    (float, array-like, array-like)
        GOW distance, transport matrix and weight vector (only if show_details==True)
    '''

    n = D.shape[0]
    m = D.shape[1]

    if len(a) == 0:
        a = np.full((n,), 1.0 / n)
    if len(b) == 0:
        b = np.full((m,), 1.0 / m)

    num_function = len(function_list)
    Y = np.empty((n*m, 1))
    V = np.empty((n*m, num_function))
    iterCount = 0
    F = np.empty((m, num_function))

    for j in range(m):
        for k in range(num_function):
            F[j][k] = compute_f(function_list[k], j)

    D_ = D

    while iterCount < maxIter:   
        iterCount = iterCount + 1

    # Optimize T
        T = ot.sinkhorn(a, b, D_, 1.0/LAMBDA2)

    # Optimize w
        index_Y = 0
        for i in range(n):
            for j in range(m):
                temp = np.sqrt(T[i][j])
                Y[index_Y][0] =  temp * i
                           
                for k in range(num_function):
                    V[index_Y][k] = temp * F[j][k]
                
                index_Y = index_Y + 1

        w_new = choose_initial_w(Y, V, num_function)
        
    # Frank-Wolfe loop
        for FW_index in range(num_FW_iteration):
            gradient = -2 * np.matmul(np.transpose(V), Y - np.matmul(V, w_new))
            min_index = np.argmin(gradient)
            s = np.zeros((num_function, 1))
            s[min_index][0] = 1
            FW_step_size = 2 / (FW_index + 2)
            w_new = w_new + FW_step_size*(s - w_new)

    # Check stopping condition
        if iterCount != 1:
            diff = (np.absolute(w_new - w_old)).max() 
            # diff = np.sqrt(np.sum((w_new - w_old) ** 2))

            if diff < epsilon:
                break

    # New cost matrix computed from the new w
        D_ = compute_new_cost2(D, w_new, F, LAMBDA1)
        w_old = w_new

    if show_details:
        return np.sum(D * T), T, w_new

    return np.sum(D * T)

def gow_sinkhorn_autoscale(a, b, D, function_list=[("polynomial", (1.0, 0)),], LAMBDA1=5, LAMBDA2=10, maxIter=15, epsilon=0.01, num_FW_iteration=100, show_details=False):
    '''Compute the GOW distance between two series with automatic scaling of
    the input functions according to the series lengths.

    Parameters
    ----------
    a : array-like, shape (dim_a,)
        Source histogram weights
    b : array-like, shape (dim_b,) or ndarray, shape (dim_b, n_hists)
        Target histogram weights
    D : array-like, shape (dim_a, dim_b)
        Loss/cost matrix
    function_list : list, optional
        Input functions used to control deformation paths
    LAMBDA1 : float, optional
        Regularization parameter for input functions
    LAMBDA2 : float, optional
        Regularization parameter for Sinkhorn
    maxIter: int, optional
        Maximum number of outer iterations (coordinate descent)
    epsilon: float, optional
        Stopping tolerance for weight updates
    num_FW_iteration: int, optional
        Number of Frank-Wolfe iterations
    show_details: bool, optional
        If True, return the GOW distance, transport matrix and weight vector

    Returns
    -------
    float
        GOW distance
    (float, array-like, array-like)
        GOW distance, transport matrix and weight vector (only if show_details==True)
    '''

    n = D.shape[0]
    m = D.shape[1]

    if len(a) == 0:
        a = np.full((n,), 1.0 / n)
    if len(b) == 0:
        b = np.full((m,), 1.0 / m)

    num_function = len(function_list)
    i_scale = n - 1
    j_scale = m - 1
    Y = np.empty((n*m, 1))
    V = np.empty((n*m, num_function))
    iterCount = 0
    F = np.empty((m, num_function))

    for j in range(m):
        for k in range(num_function):
            F[j][k] = compute_f_scale(function_list[k], j, i_scale, j_scale)

    w_0 = np.zeros(num_function)
    w_0[np.random.randint(num_function)] = 1
    D_ = compute_new_cost(D, w_0, F, LAMBDA1)

    while iterCount < maxIter:
        iterCount = iterCount + 1

    # Optimize T
        T = ot.sinkhorn(a, b, D_, 1.0/LAMBDA2)

    # Optimize w
        index_Y = 0
        for i in range(n):
            for j in range(m):
                temp = np.sqrt(T[i][j])
                Y[index_Y][0] =  temp * i
                           
                for k in range(num_function):
                    V[index_Y][k] = temp * F[j][k]
                
                index_Y = index_Y + 1

        w_new = choose_initial_w(Y, V, num_function)
        
    # Frank-Wolfe loop
        for FW_index in range(num_FW_iteration):
            gradient = -2 * np.matmul(np.transpose(V), Y - np.matmul(V, w_new))
            min_index = np.argmin(gradient)
            s = np.zeros((num_function, 1))
            s[min_index][0] = 1
            FW_step_size = 2 / (FW_index + 2)
            w_new = w_new + FW_step_size*(s - w_new)

    # New cost matrix computed from the new w
        D_ = compute_new_cost(D, w_new, F, LAMBDA1)

    # Check stopping condition
        if iterCount != 1:
            diff = (np.absolute(w_new - w_old)).max() 

            if diff < epsilon:
                break

        w_old = w_new

    if show_details:
        return np.sum(D * T), T, w_new

    return np.sum(D * T)

def gow_sinkhorn_autoscale_fixed(a, b, D, LAMBDA1=10, LAMBDA2=5, maxIter=15, epsilon=0.01, num_FW_iteration=100, show_details=False):
    '''
    Tính khoảng cách GOW giữa hai chuỗi.
    Nếu ma trận chi phí D có kích thước n x m,
    ma trận vận chuyển kết quả có kích thước n x m.
    Không cần các hàm đầu vào vì hàm này
    sử dụng 5 hàm đơn điệu cố định.

    Tham số
    ----------
    a : array-like, shape (dim_a,)
        Trọng số mẫu ở miền nguồn
    b : array-like, shape (dim_b,) hoặc ndarray, shape (dim_b, n_hists)
        Mẫu ở miền đích
    D : array-like, shape (dim_a, dim_b)
        Ma trận mất mát
    LAMBDA1 : float, tùy chọn
        Tham số điều chuẩn cho các hàm đầu vào
    LAMBDA2 : float, tùy chọn
        Tham số điều chuẩn cho Sinkhorn
    maxIter: int, tùy chọn
        Số vòng lặp tối đa cho vòng lặp chính (Coordinate Descent)
    epsilon: float, tùy chọn
        Ngưỡng dừng theo sai số
    num_FW_iteration: int, tùy chọn
        Số vòng lặp Frank-Wolfe
    show_details: bool, tùy chọn
        Trả về khoảng cách GOW, ma trận vận chuyển và vector trọng số nếu True

    Trả về
    -------
    float : khoảng cách GOW
    float, array-like, array-like:
        Khoảng cách GOW, ma trận vận chuyển và vector trọng số (chỉ trả về nếu show_details==True)
    '''

    n = D.shape[0]
    m = D.shape[1]
    i_scale = n - 1
    j_scale = m - 1

    if len(a) == 0:
        a = np.full((n,), 1.0 / n)
    if len(b) == 0:
        b = np.full((m,), 1.0 / m)

    # 5 hàm đơn điệu cố định
    func1 = ('polynomial_with_degree', (1.0, 1.0, 0.05))
    func2 = ('polynomial_with_degree', (1.0, 1.0, 0.28))
    func3 = ("polynomial", (1.0, 0)) 
    func4 = ('polynomial_with_degree', (1.0, 1.0, 3.2))
    func5 = ('polynomial_with_degree', (1.0, 1.0, 20))

    function_list = [func1, func2, func3, func4, func5]
    num_function = len(function_list)
    Y = np.empty((n*m, 1))
    V = np.empty((n*m, num_function))
    iterCount = 0
    F = np.empty((m, num_function))

    for j in range(m):
        for k in range(num_function):
            F[j][k] = compute_f_scale(function_list[k], j, i_scale, j_scale)

    w_0 = np.array([0, 0, 1, 0, 0])
    D_ = compute_new_cost(D, w_0, F, LAMBDA1)

    while iterCount < maxIter: 
        iterCount = iterCount + 1

        # Tối ưu T
        T = ot.sinkhorn(a, b, D_, 1.0/LAMBDA2)

        # Tối ưu w
        index_Y = 0
        for i in range(n):
            for j in range(m):
                temp = np.sqrt(T[i][j])
                Y[index_Y][0] =  temp * i
                           
                for k in range(num_function):
                    V[index_Y][k] = temp * F[j][k]
                
                index_Y = index_Y + 1

        w_new = choose_initial_w(Y, V, num_function)
        
        # Vòng lặp Frank-Wolfe
        for FW_index in range(num_FW_iteration):
            gradient = -2 * np.matmul(np.transpose(V), Y - np.matmul(V, w_new))
            min_index = np.argmin(gradient)
            s = np.zeros((num_function, 1))
            s[min_index][0] = 1
            FW_step_size = 2 / (FW_index + 2)
            w_new = w_new + FW_step_size*(s - w_new)

        # Ma trận chi phí mới từ w mới
        D_ = compute_new_cost(D, w_new, F, LAMBDA1)

        # Kiểm tra điều kiện dừng
        if iterCount != 1:
            diff = (np.absolute(w_new - w_old)).max() 

            if diff < epsilon:
                break

        w_old = w_new

    if show_details:
        return np.sum(D * T), T, w_new

    return np.sum(D * T)


In [ ]:
import numpy as np
import ot

def _get_xp(use_gpu="auto"):
    if use_gpu is False:
        return np, False
    try:
        import cupy as cp
        # ensure CUDA is usable (otherwise fall back to numpy)
        try:
            _ = cp.cuda.runtime.getDeviceCount()
            return cp, True
        except Exception:
            if use_gpu is True:
                raise
            return np, False
    except Exception:
        if use_gpu is True:
            raise
        return np, False


def opw_distance_series(
    x, y,
    lambda1=1.0,
    lambda2=0.1,
    sigma=0.1,
    use_gpu="auto",
    max_iter=1000,
    tol=5e-3,
):
    xp, on_gpu = _get_xp(use_gpu)

    x = xp.asarray(x, dtype=xp.float64)
    y = xp.asarray(y, dtype=xp.float64)
    if x.ndim == 1: x = x[:, None]
    if y.ndim == 1: y = y[:, None]

    Nx, _ = x.shape
    Ny, _ = y.shape

    a = xp.full((Nx,), 1.0 / Nx, dtype=xp.float64)
    b = xp.full((Ny,), 1.0 / Ny, dtype=xp.float64)

    # Ground cost D_ij = ||x_i - y_j||^2
    x2 = xp.sum(x * x, axis=1)[:, None]
    y2 = xp.sum(y * y, axis=1)[None, :]
    D = x2 + y2 - 2.0 * (x @ y.T)
    D = xp.maximum(D, 0.0)

    # Time geometry
    i_norm = xp.arange(Nx, dtype=xp.float64)[:, None] / Nx
    j_norm = xp.arange(Ny, dtype=xp.float64)[None, :] / Ny
    diff = i_norm - j_norm

    S = diff**2 + 1.0
    
    # Total cost: D + lambda1*S
    M_base = D + lambda1 * S
    
    # Chuẩn hóa cost giống TAOT để tăng tốc Sinkhorn
    med = xp.median(M_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0
    M = M_base / med
    
    # Sinkhorn đơn giản (nhanh hơn bregman_log_projection_batch)
    T = ot.sinkhorn(a, b, M, reg=lambda2, numItermax=max_iter, stopThr=tol)

    dist = xp.sum(T * M_base)
    return float(dist.get()) if on_gpu else float(dist)

In [ ]:
import numpy as np
import ot

try:
    import cupy as cp
except ImportError:
    cp = None


def asw_distance(
    A,
    B,
    lam=10.0,          # như TAOT: reg = 1 / lam cho Sinkhorn
    auto_weight=True,  # bật/tắt auto-weight cho 3 term
    w_spatial=1.0,     # weight cho spatial khi không auto
    w_order=1.0,       # weight cho order khi không auto
    w_struct=1.0,      # weight cho structural khi không auto
    tolerance=5e-3,
):
    """
    ASW distance (phiên bản gần đúng theo tinh thần Auto-weighted Sequential Wasserstein),
    hỗ trợ GPU nếu A hoặc B là cupy.ndarray.

    Ý tưởng:
    - C_spatial(i,j) = ||x_i - y_j||^2
    - C_order(i,j)   = (t_i - s_j)^2  với t_i, s_j là vị trí chuẩn hoá trong [0,1]
    - C_struct(i,j)  = ||gA_i - gB_j||^2, trong đó gA_i, gB_j là "gradient/cấu trúc lân cận"
      (chênh lệch giữa phần tử hiện tại và phần tử trước nó).

    - Tổng cost: C = w_s * C_spatial + w_o * C_order + w_n * C_struct
      với bộ weight có thể:
        + auto_weight=True: w_s, w_o, w_n lấy tự động từ dữ liệu (xấp xỉ ASW gốc),
        + auto_weight=False: dùng w_spatial, w_order, w_struct do người dùng cung cấp.

    Sau đó giải Sinkhorn như TAOT/POW:
      reg = 1 / lam
      asw = sum_{i,j} P_ij * C_ij
    """

    # 0) Backend: NumPy (CPU) hoặc CuPy (GPU)
    use_gpu = (cp is not None) and (
        isinstance(A, (cp.ndarray,)) or isinstance(B, (cp.ndarray,))
    )
    xp = cp if use_gpu else np

    # 1) Đưa dữ liệu về đúng backend, đảm bảo shape (n,d)
    A = xp.asarray(A, dtype=xp.float64)
    B = xp.asarray(B, dtype=xp.float64)

    if A.ndim == 1:
        A = A[:, None]
    if B.ndim == 1:
        B = B[:, None]

    n, dA = A.shape
    m, dB = B.shape
    if n == 0 or m == 0:
        raise ValueError("empty")
    if dA != dB:
        raise ValueError("dim mismatch")

    # ===== 2) C_spatial: khoảng cách đặc trưng =====
    diff = A[:, None, :] - B[None, :, :]
    C_spatial = xp.sum(diff * diff, axis=2)  # (n, m)

    # ===== 3) C_order: khoảng cách vị trí (index) =====
    # Chuẩn hoá index về [0,1]
    if n > 1:
        t = xp.linspace(0.0, 1.0, n)
    else:
        t = xp.zeros(1, dtype=xp.float64)

    if m > 1:
        s = xp.linspace(0.0, 1.0, m)
    else:
        s = xp.zeros(1, dtype=xp.float64)

    C_order = (t[:, None] - s[None, :]) ** 2  # (n, m)

    # ===== 4) C_struct: khoảng cách cấu trúc/gradient =====
    # Gradient đơn giản: gA[i] = A[i] - A[i-1], với gA[0] = 0
    gA = xp.zeros_like(A)
    if n > 1:
        gA[1:] = A[1:] - A[:-1]

    gB = xp.zeros_like(B)
    if m > 1:
        gB[1:] = B[1:] - B[:-1]

    gdiff = gA[:, None, :] - gB[None, :, :]
    C_struct = xp.sum(gdiff * gdiff, axis=2)  # (n, m)

    # ===== 5) Auto-weight hay dùng weight tay =====
    def _mean_safe(M):
        mu = xp.mean(M)
        if not xp.isfinite(mu) or float(mu) == 0.0:
            return 1.0
        return float(mu)

    if auto_weight:
        # Trọng số tỉ lệ nghịch với độ lớn trung bình của từng term:
        # term nào lớn quá -> weight nhỏ lại, để các term cân bằng hơn.
        mu_s = _mean_safe(C_spatial)
        mu_o = _mean_safe(C_order)
        mu_n = _mean_safe(C_struct)

        ws = 1.0 / mu_s
        wo = 1.0 / mu_o
        wn = 1.0 / mu_n
    else:
        ws = float(w_spatial)
        wo = float(w_order)
        wn = float(w_struct)

    C_base = ws * C_spatial + wo * C_order + wn * C_struct  # (n, m)

    # ===== 6) Chuẩn hoá cost cho Sinkhorn (giống TAOT/POW) =====
    med = xp.median(C_base)
    if not xp.isfinite(med) or float(med) <= 0.0:
        med = 1.0

    C = C_base / med

    # ===== 7) Khối lượng đều + Sinkhorn =====
    a = xp.full(n, 1.0 / n, dtype=xp.float64)
    b = xp.full(m, 1.0 / m, dtype=xp.float64)

    reg = 1.0 / lam

    P = ot.sinkhorn(a, b, C, reg, stopThr=tolerance)

    # ===== 8) ASW distance =====
    distance_raw = float(xp.sum(P * C_base))
    return distance_raw


In [ ]:
# otsw_api.py  — RAGGED-FRIENDLY
import numpy as np
from dataclasses import dataclass
from typing import List, Tuple, Optional, Dict, Sequence, Union
import heapq

# (Optional) Use SciPy to accelerate SpMM; works without it as well
try:
    import scipy.sparse as sp
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

BIG = 1e12

# =========================
# 0) HELPERS (ragged / dense)
# =========================
def _as_ragged_list(M: Union[np.ndarray, Sequence[np.ndarray]]) -> Tuple[List[np.ndarray], int]:
    """
    Normalize input to a list of arrays (n_i, d).
    Returns (list_seq, d)
    """
    if isinstance(M, np.ndarray):
        if M.ndim != 3:
            raise ValueError("If ndarray, expect shape (m, n, d).")
        m, n, d = M.shape
        seqs = [M[i] for i in range(m)]
        return seqs, d
    # list/tuple các chuỗi (n_i, d)
    seqs = []
    d = None
    for i, xi in enumerate(M):
        xi = np.asarray(xi, dtype=float)
        if xi.ndim != 2:
            raise ValueError(f"Sequence {i} must have shape (n_i, d).")
        if d is None:
            d = xi.shape[1]
        elif xi.shape[1] != d:
            raise ValueError("All sequences must have the same feature dimension d.")
        seqs.append(xi)
    if d is None:
        raise ValueError("Empty sequence list.")
    return seqs, d

def _linearize_points_ragged(M: Union[np.ndarray, Sequence[np.ndarray]]):
    """
    Ragged support: 
      - P: (N, d) concatenated points
      - Sidx: (N,) series id
      - Tpos: (N,) normalized time in [0,1) for each point (i / n_i)
      - lengths: (m_seq,) length of each series
      - d: number of channels
    """
    seqs, d = _as_ragged_list(M)
    m_seq = len(seqs)
    lengths = np.array([xi.shape[0] for xi in seqs], dtype=int)
    # concatenate points
    P = np.vstack(seqs) if m_seq > 0 else np.zeros((0, d))
    # series id
    Sidx = np.repeat(np.arange(m_seq, dtype=int), lengths)
    # normalized time position (avoid endpoint=1 to prevent collision at 1.0)
    Tpos_list = [ (np.arange(n_i, dtype=float) / max(n_i,1)) for n_i in lengths ]
    Tpos = np.concatenate(Tpos_list) if m_seq > 0 else np.zeros((0,), float)
    return P, Sidx, Tpos, m_seq, lengths, d

def _pairwise_sqdist(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    """
    'Hybrid' distance:
      Euclid^2 on features (except last column)  +  |orderA - orderB| (last column)
    """
    assert A.ndim == 2 and B.ndim == 2, "Expect 2D arrays"
    assert A.shape[1] == B.shape[1], "Dim mismatch"
    D = A.shape[1]
    if D == 0:
        return np.zeros((A.shape[0], B.shape[0]), dtype=float)
    if D == 1:
        a_ord = A[:, 0]; b_ord = B[:, 0]
        return np.abs(a_ord[:, None] - b_ord[None, :])
    Af = A[:, :-1]; Bf = B[:, :-1]
    aa = (Af * Af).sum(1)[:, None]
    bb = (Bf * Bf).sum(1)[None, :]
    Dsq = np.clip(aa + bb - 2 * (Af @ Bf.T), 0.0, None)
    a_ord = A[:, -1]; b_ord = B[:, -1]
    Pen = np.abs(a_ord[:, None] - b_ord[None, :])
    return Dsq + Pen

@dataclass
class _Node:
    idx: np.ndarray
    height: float
    left: Optional[int]
    right: Optional[int]
    parent: Optional[int]
    is_leaf: bool

@dataclass
class OTSWModel:
    # shared runtime fields
    P: np.ndarray                  # (N, d_aug) nếu TamLe, hoặc (N, d) nếu Banded
    Sidx: np.ndarray               # (N,) id chuỗi
    Tpos: np.ndarray               # (N,) thời gian chuẩn hoá (0..1)
    lengths: np.ndarray            # (m_seq,)
    m_seq: int
    d: int                         # số kênh gốc (chưa augment)
    nodes: List[_Node]
    leaf_ids: List[int]
    leaf_index_map: Dict[int, int]
    edges: List[Tuple[int, int, float]]     # (parent, child, w_e)
    S_edge_leaf: object                     # (E, L) dense hoặc sp.csr_matrix
    centroids: np.ndarray                   # (num_nodes, D_aug)
    # meta
    mode: str                               # "tamle" | "banded"
    lam_time: float = 0.0
    lam_idx: float = 0.0
    W: float = 0.0                           # band width (tỉ lệ 0..1 cho ragged)
    # caches
    point_leaf: Optional[np.ndarray] = None  # (N,)
    H: Optional[np.ndarray] = None           # (L, m_seq)
    M: Optional[np.ndarray] = None           # (E, m_seq)
    w: Optional[np.ndarray] = None           # (E,)

# =========================
# 1) BOX TREE + GONZALEZ
# =========================
class _KDBoxTree:
    def __init__(self, leaf_size=64, max_depth=24):
        self.leaf_size = leaf_size
        self.max_depth = max_depth
        self.boxes = []
        self.X = None

    def _build(self, idx, depth):
        X = self.X[idx]
        c = X.mean(axis=0)
        r = float(np.sqrt(((X - c) ** 2).sum(1).max())) if X.shape[0] else 0.0
        bid = len(self.boxes)
        self.boxes.append({"idx": idx, "c": c, "r": r, "L": None, "R": None, "leaf": False})
        if idx.size <= self.leaf_size or depth >= self.max_depth or r == 0.0:
            self.boxes[bid]["leaf"] = True
            return bid
        var = X.var(axis=0)
        d = int(np.argmax(var))
        med = np.median(X[:, d])
        mask = X[:, d] <= med
        if mask.all() or (~mask).all():
            mid = idx.size // 2
            Lidx = idx[:mid]; Ridx = idx[mid:]
        else:
            Lidx = idx[mask]; Ridx = idx[~mask]
        L = self._build(Lidx, depth + 1)
        R = self._build(Ridx, depth + 1)
        self.boxes[bid]["L"] = L; self.boxes[bid]["R"] = R
        return bid

    def fit(self, X):
        self.X = X
        self.boxes = []
        self._build(np.arange(X.shape[0]), 0)

def _bounds_box(box, centers):
    if centers.size == 0: return 0.0, float("inf")
    d = np.sqrt(((centers - box["c"][None, :]) ** 2).sum(1))
    dmin = float(d.min())
    r = box["r"]
    return max(0.0, dmin - r), dmin + r

def _farthest_point_by_boxes(X, centers, kdt: _KDBoxTree, gap_tol=1e-6):
    if centers.size == 0: return 0, 0.0
    heap = []
    L0, U0 = _bounds_box(kdt.boxes[0], centers)
    heapq.heappush(heap, (-U0, 0, L0))
    best_idx, best_val = None, -1.0
    while heap:
        negU, bid, Lb = heapq.heappop(heap)
        Ub = -negU
        L2, U2 = _bounds_box(kdt.boxes[bid], centers)
        if U2 < Ub - 1e-12 or L2 > Lb + 1e-12:
            heapq.heappush(heap, (-U2, bid, L2)); continue
        if best_val >= U2 - 1e-15: break
        box = kdt.boxes[bid]
        if box["leaf"] or (U2 - L2) <= gap_tol:
            pts = kdt.X[box["idx"]]
            D = _pairwise_sqdist(pts, centers)  # KHÔNG sqrt
            dmin = D.min(axis=1)
            imax = int(np.argmax(dmin)); val = float(dmin[imax])
            if val > best_val: best_val, best_idx = val, int(box["idx"][imax])
            continue
        for child in (box["L"], box["R"]):
            Lc, Uc = _bounds_box(kdt.boxes[child], centers)
            heapq.heappush(heap, (-Uc, child, Lc))
    return best_idx, best_val

def _gonzalez_box_nlogk(X: np.ndarray, k: int, seed: int,
                        box_leaf_size=64, box_max_depth=24, gap_tol=1e-6):
    rng = np.random.default_rng(seed)
    n = X.shape[0]; assert 1 <= k <= n
    kdt = _KDBoxTree(leaf_size=box_leaf_size, max_depth=box_max_depth); kdt.fit(X)
    i0 = int(rng.integers(0, n)); centers = X[i0:i0+1]; C = [i0]
    for _ in range(1, k):
        idx, _ = _farthest_point_by_boxes(X, centers, kdt, gap_tol)
        C.append(idx); centers = X[np.array(C)]
    return np.array(C, dtype=int)

# =========================
# 1.1) ROUTING & PRECOMPUTE
# =========================
def _route_all_points_vectorized(model: OTSWModel) -> np.ndarray:
    N = model.P.shape[0]
    leaf_of_point = np.empty(N, dtype=np.int32)
    stack = [(0, np.arange(N, dtype=np.int32))]
    nodes = model.nodes; C = model.centroids; P = model.P
    while stack:
        nid, idxs = stack.pop()
        nd = nodes[nid]
        if nd.is_leaf:
            j = model.leaf_index_map[nid]; leaf_of_point[idxs] = j; continue
        L = nd.left; R = nd.right
        X = P[idxs]
        child_centroids = np.vstack([C[L], C[R]])
        d_lr = _pairwise_sqdist(X, child_centroids)
        go_left = d_lr[:, 0] <= d_lr[:, 1]
        if go_left.any():    stack.append((L, idxs[go_left]))
        if (~go_left).any(): stack.append((R, idxs[~go_left]))
    return leaf_of_point

def _precompute_H_M(model: OTSWModel):
    m_seq = model.m_seq
    N = model.P.shape[0]
    L = len(model.leaf_ids)
    E = len(model.edges)
    # 1) route tất cả điểm -> lá
    point_leaf = _route_all_points_vectorized(model)  # (N,)
    model.point_leaf = point_leaf
    # 2) H (L, m_seq) — histogram mỗi chuỗi
    H = np.zeros((L, m_seq), dtype=np.float32)
    for s in range(m_seq):
        mask = (model.Sidx == s)
        if not np.any(mask): continue
        counts = np.bincount(point_leaf[mask], minlength=L).astype(np.float32)
        tot = counts.sum()
        if tot > 0: counts /= tot
        H[:, s] = counts
    model.H = H
    # 3) S_edge_leaf -> CSR (nếu có SciPy) và M = S @ H
    if _HAS_SCIPY:
        SpS = sp.csr_matrix(model.S_edge_leaf)
        model.S_edge_leaf = SpS
        M = (SpS @ H).astype(np.float32)  # (E, m_seq)
    else:
        M = (model.S_edge_leaf @ H).astype(np.float32)
    model.M = M
    # 4) Trọng số cạnh
    model.w = np.array([we for _, _, we in model.edges], dtype=np.float32)

# =========================
# 1.2) OTSW — TAM LE (ragged OK)
# =========================
def _augment_points(seq: np.ndarray, lam_time: float) -> np.ndarray:
    n = seq.shape[0]
    t = (np.arange(n, dtype=float) / max(n, 1))[:, None] * np.sqrt(lam_time)
    return np.hstack([seq, t])

def build_otsw_tamle(
    M: Union[np.ndarray, Sequence[np.ndarray]],
    lam_time: float = 5.0,
    leaf_size: int = 16,
    max_depth: int = 20,
    seed: int = 0,
    k_split: int = 2,
    box_leaf_size: int = 64,
    box_max_depth: int = 24,
) -> OTSWModel:
    """
    Xây cây global theo TamLe (augment theo thời gian chuẩn hoá → ragged friendly).
    """
    P_raw, Sidx, Tpos, m_seq, lengths, d = _linearize_points_ragged(M)
    # augment từng chuỗi rồi ghép
    P_aug_list = []
    start = 0
    for s in range(m_seq):
        n_i = lengths[s]
        seq = P_raw[start:start+n_i]
        P_aug_list.append(_augment_points(seq, lam_time))
        start += n_i
    P_aug = np.vstack(P_aug_list) if P_aug_list else np.zeros((0, d+1))
    # build tree
    nodes: List[_Node] = []; leaf_ids: List[int] = []
    def _hybrid_radius(X):
        if X.shape[0] <= 1: return 0.0
        if X.shape[0] > 1024:
            I = np.random.default_rng(0).choice(X.shape[0], 1024, replace=False); Y = X[I]
        else: Y = X
        j0 = 0; d0 = _pairwise_sqdist(Y, Y[j0:j0+1]).reshape(-1); j1 = int(np.argmax(d0))
        d1 = _pairwise_sqdist(Y, Y[j1:j1+1]).reshape(-1); return 0.5 * float(d1.max())
    def build(idx: np.ndarray, depth: int, parent: Optional[int], seed_: int) -> int:
        Xsub = P_aug[idx]; h = _hybrid_radius(Xsub)
        nid = len(nodes); nodes.append(_Node(idx, h, None, None, parent, False))
        if idx.size <= leaf_size or depth >= max_depth or h == 0.0:
            nodes[nid].is_leaf = True; leaf_ids.append(nid); return nid
        C = _gonzalez_box_nlogk(Xsub, k=k_split, seed=seed_,
                                box_leaf_size=box_leaf_size, box_max_depth=box_max_depth)
        centers = Xsub[C]
        lab = np.argmin(_pairwise_sqdist(Xsub, centers), axis=1)
        if k_split == 2:
            left_idx = idx[lab == 0]; right_idx = idx[lab != 0]
        else:
            cnt = np.bincount(lab, minlength=k_split); main = int(np.argmax(cnt))
            left_idx = idx[lab == main]; right_idx = idx[lab != main]
        if left_idx.size == 0 or right_idx.size == 0:
            mid = idx.size // 2; left_idx = idx[:mid]; right_idx = idx[mid:]
        L = build(left_idx, depth+1, nid, seed_+1); R = build(right_idx, depth+1, nid, seed_+2)
        nodes[nid].left, nodes[nid].right = L, R; return nid
    _ = build(np.arange(P_aug.shape[0]), 0, None, seed)
    # edges & structures
    edges = []
    for cid, nd in enumerate(nodes):
        if nd.parent is not None:
            p = nodes[nd.parent]; w = max(0.0, p.height - nd.height)
            edges.append((nd.parent, cid, w))
    leaf_index_map = {nid: i for i, nid in enumerate(leaf_ids)}
    E, L = len(edges), len(leaf_ids)
    S_edge_leaf = np.zeros((E, L), dtype=np.float32)
    def collect_leaves(nid, out):
        nd = nodes[nid]
        if nd.is_leaf: out.append(nid); return
        if nd.left is not None: collect_leaves(nd.left, out)
        if nd.right is not None: collect_leaves(nd.right, out)
    for e, (pid, cid, _) in enumerate(edges):
        leaves = []; collect_leaves(cid, leaves)
        for ln in leaves:
            j = leaf_index_map[ln]; S_edge_leaf[e, j] = 1.0
    centroids = np.vstack([P_aug[nd.idx].mean(axis=0) for nd in nodes])
    model = OTSWModel(
        P=P_aug, Sidx=Sidx, Tpos=Tpos, lengths=lengths, m_seq=m_seq, d=d,
        nodes=nodes, leaf_ids=leaf_ids, leaf_index_map=leaf_index_map,
        edges=edges, S_edge_leaf=S_edge_leaf, centroids=centroids,
        mode="tamle", lam_time=lam_time
    )
    _precompute_H_M(model)
    return model

# =========================
# 3) DISTANCE APIs
# =========================
def otsw_between_series_fast(model: OTSWModel, s_ref: int, s_cmp: int) -> float:
    """
    OTSW(s_ref, s_cmp) với cache:
      cost = sum_e w_e * |M[e, s_ref] - M[e, s_cmp]|
    """
    w = model.w; M = model.M
    diff = np.abs(M[:, s_ref] - M[:, s_cmp])
    return float((w * diff).sum())

def otsw_between_series(model: OTSWModel, s_ref: int, s_cmp: int, p: int = 1) -> float:
    assert p == 1, "Hiện tại hỗ trợ p=1 (W1    cây)."
    return otsw_between_series_fast(model, s_ref, s_cmp)

In [ ]:
import os
from os.path import basename, join
import sys
import time

import joblib
import numpy as np
import ot
import pandas as pd

from sklearn import neighbors
from sklearn.metrics import accuracy_score, average_precision_score
from tqdm import tqdm

# để import các hàm distance khác nếu bạn đặt trong src/
sys.path.append('src')
# from your_module import dtw_distance_series, taot_distance, gow_sinkhorn_autoscale
import cupy as cp

# =========================
#  Loader UCR từ tslearn
# =========================
def load_ucr_dataset_tsl(data_dir, dataset_name):
    """
    Loads train and test data using tslearn's UCR/UEA loader.
    data_dir hiện không sử dụng nhưng giữ lại cho tương thích.
    """
    from tslearn.datasets import UCR_UEA_datasets
    X_train, y_train, X_test, y_test = UCR_UEA_datasets().load_dataset(dataset_name)

    print("Successfully loaded dataset:", dataset_name)
    print("Size of train data:", len(y_train))
    print("Size of test data:", len(y_test))

    return X_train, y_train, X_test, y_test


# =========================
#  Loader Human Action datasets
# =========================
def load_human_action_dataset(data_dir, dataset_name):
    """
    Loads train and test data from Human Action datasets stored in joblib pickle files.
    Supports: MSRAction3D, Weizmann, SpokenArabicDigit
    """
    import os
    from os.path import join
    
    data_path = join(data_dir, dataset_name)
    
    print(f"Loading {dataset_name} from {data_path}...")
    
    X_train = joblib.load(join(data_path, "X_train.pkl"))
    y_train = joblib.load(join(data_path, "y_train.pkl"))
    X_test = joblib.load(join(data_path, "X_test.pkl"))
    y_test = joblib.load(join(data_path, "y_test.pkl"))
    
    # Convert to numpy arrays if they are lists
    if isinstance(X_train, list):
        X_train = np.array(X_train, dtype=object)
    if isinstance(y_train, list):
        y_train = np.array(y_train)
    if isinstance(X_test, list):
        X_test = np.array(X_test, dtype=object)
    if isinstance(y_test, list):
        y_test = np.array(y_test)
    
    print(f"Successfully loaded dataset: {dataset_name}")
    print(f"Size of train data: {len(y_train)}")
    print(f"Size of test data: {len(y_test)}")
    
    return X_train, y_train, X_test, y_test


# =========================
#  Tính MAP đúng kiểu paper
# =========================
def compute_map_knn_precomputed(X_computed, X_test_computed,
                                y_train, y_test, k=1):
    """
    Tính MAP cho k-NN (metric='precomputed') theo mô tả trong paper:

      - Fit k-NN với số láng giềng k.
      - predict_proba trên test -> score cho từng lớp.
      - Với mỗi lớp c:
          AP_c = average_precision_score(1_{y_test==c}, score_c)
      - MAP = trung bình AP_c trên tất cả các lớp.

    Trả về MAP dạng phần trăm [%].
    """
    clf = neighbors.KNeighborsClassifier(
        n_neighbors=k,
        metric="precomputed",
        weights="uniform",
    )
    clf.fit(X_computed, y_train)

    proba = clf.predict_proba(X_test_computed)   # shape (n_test, n_classes)
    classes = clf.classes_

    aps = []
    for c_idx, c in enumerate(classes):
        y_true_c = (y_test == c).astype(int)
        if np.sum(y_true_c) == 0:
            # không có sample lớp c trong test -> bỏ qua
            continue

        y_score_c = proba[:, c_idx]

        # Nếu mọi score y hệt nhau thì PR curve thoái hoá, coi AP = 0
        if np.all(y_score_c == y_score_c[0]):
            aps.append(0.0)
        else:
            ap_c = average_precision_score(y_true_c, y_score_c)
            aps.append(ap_c)

    if len(aps) == 0:
        return 0.0

    return float(np.mean(aps) * 100.0)


# =========================
#  Hàm chính run_knn (nhiều lần)
# =========================
def run_knn(datapath, datatype, alg,
            normalize_cost_matrix=True,
            cost_metric="minkowski",
            num_neighbor_list=[1, 3, 5, 10, 15, 30],
            num_runs=5):
    """
    Run k-NN với precomputed distances, lặp nhiều lần:

      - Mỗi lần:
          + Tính ma trận khoảng cách train–train, test–train.
          + Chạy k-NN với nhiều k, lấy best accuracy (theo k).
          + Tính MAP (dùng k = best_k hoặc k cố định tuỳ chọn).
      - Sau num_runs lần:
          + Lấy mean và variance cho:
                accuracy, map, runtime.

    Ghi vào Excel với các cột:
      dataset, accuracy_mean, accuracy_var, map_mean, map_var,
      runtime_mean, runtime_var
    """
    # 1. Load dữ liệu (chỉ 1 lần)
    if datatype == "UCR_TSL":
        X_train, y_train, X_test, y_test = load_ucr_dataset_tsl("../data/UCR", datapath)
    elif datatype == "Human_Actions":
        X_train, y_train, X_test, y_test = load_human_action_dataset("/kaggle/input/dataset/data/Human_Actions", datapath)
    else:
        raise ValueError(f"Unknown datatype: {datatype}")

    # Downsample CinCECGTorso và MixedShapesSmallTrain xuống 300 mẫu
    if datapath in ["CinCECGTorso", "MixedShapesSmallTrain"]:
        X_all = np.concatenate([X_train, X_test], axis=0)
        y_all = np.concatenate([y_train, y_test], axis=0)
        
        if len(X_all) > 300:
            rng = np.random.default_rng(0)
            idx = rng.choice(len(X_all), size=300, replace=False)
            X_all = X_all[idx]
            y_all = y_all[idx]
            print(f"[DOWNSAMPLE] {datapath}: {len(X_train)+len(X_test)} → 300 samples")
        
        # Chia lại train/test 70/30
        n_train = int(0.7 * len(X_all))
        X_train, y_train = X_all[:n_train], y_all[:n_train]
        X_test, y_test = X_all[n_train:], y_all[n_train:]
    
    # Downsample SpokenArabicDigit to 10%
    if datapath == "SpokenArabicDigit":
        X_all = np.concatenate([X_train, X_test], axis=0)
        y_all = np.concatenate([y_train, y_test], axis=0)
        
        target_size = int(len(X_all) * 0.1)
        rng = np.random.default_rng(0)
        idx = rng.choice(len(X_all), size=target_size, replace=False)
        X_all = X_all[idx]
        y_all = y_all[idx]
        
        print(f"[DOWNSAMPLE] {datapath}: {len(X_train)+len(X_test)} → {target_size} samples (10%)")
        
        # Split into train/test 70/30
        n_train = int(0.7 * len(X_all))
        X_train, y_train = X_all[:n_train], y_all[:n_train]
        X_test, y_test = X_all[n_train:], y_all[n_train:]

    train_len = len(y_train)
    test_len = len(y_test)

    # Danh sách để gom kết quả của nhiều lần chạy
    acc_list = []
    map_list = []
    time_list = []

    # ====== Lặp nhiều lần ======
    for run_idx in range(num_runs):
        print(f"\n========== Run {run_idx + 1}/{num_runs} for {alg} on {datapath} ==========")
        t0 = time.time()

        # 2. Khởi tạo ma trận khoảng cách cho lần chạy này
        X_computed = np.zeros((train_len, train_len), dtype=float)      # train–train
        X_test_computed = np.empty((test_len, train_len), dtype=float)  # test–train
        
           
        # 2) Build OTSW model (một lần)
        if alg == "OTSW":
            # 1) Gom chuỗi theo đúng thứ tự: test trước, rồi train (m = test_len + train_len)
            sequences = [np.asarray(X_test[i],  dtype=float) for i in range(test_len)] + \
                        [np.asarray(X_train[j], dtype=float) for j in range(train_len)]
    
            n_trees = 5
            
            # Khởi tạo ma trận kết quả tích lũy (Accumulator)
            # X_test_computed cần được khởi tạo bằng 0 để cộng dồn
            # Giả sử shape là (test_len, train_len) như logic cũ
            dist_accumulator = np.zeros((test_len, train_len), dtype=float)

            for t in range(n_trees):
                # QUAN TRỌNG: Thay đổi seed cho mỗi cây để tạo sự đa dạng (diversity)
                # Nếu giữ nguyên seed, 5 cây sẽ y hệt nhau -> trung bình vô nghĩa.
                current_seed = (run_idx * 100) + t 

                # 2) Build OTSW model (cho cây thứ t)
                model_otsw = build_otsw_tamle(
                    sequences,
                    lam_time=5.0,       # User parameter
                    leaf_size=16,       # User parameter
                    max_depth=20,       # User parameter
                    seed=current_seed,  # Seed thay đổi theo t
                    k_split=2,          # User parameter
                )

                # 3) Tính distance test-vs-train cho cây t:
                m_total = test_len + train_len
                M_edge_mass = model_otsw.M          # (E, m_total)
                w = model_otsw.w.reshape(-1, 1)     # (E, 1)

                for i in range(test_len):
                    # Vector hoá: khoảng cách từ chuỗi i (test) đến tất cả chuỗi
                    dist_all = (w * np.abs(M_edge_mass[:, i:i+1] - M_edge_mass)).sum(axis=0)
                    
                    # Cộng dồn vào kết quả tổng (chỉ lấy phần train)
                    dist_accumulator[i, :] += dist_all[test_len : test_len + train_len]

            # 4) Lấy trung bình
            X_test_computed = dist_accumulator / n_trees
    
        else:
            # 3. Hàm nội bộ tính distance cho một cặp chuỗi
            def _pair_distance(x, y):
                if alg == "DTW":
                    return dtw_distance_series(x, y)
                elif alg == "TAOT":
                    return taot_distance(x, y)
                elif alg == "GOW":
                    C = ot.dist(x, y, metric=cost_metric)
                    if normalize_cost_matrix:
                        maxC = C.max()
                        if maxC > 0:
                            C = C / maxC
                    return gow_sinkhorn_autoscale([], [], C)
                elif alg == "POW":
                    return pow_distance(x, y)
                elif alg == "ASW":
                    return asw_distance(x, y, lam=10.0, auto_weight=True)
                elif alg == "TCOT":
                    x_gpu = (x)
                    y_gpu = (y)
                    return tcot_distance_series(x_gpu, y_gpu)
                elif alg == "OPW":
                    x_gpu = (x)
                    y_gpu = (y)
                    return opw_distance_series(
                        x_gpu, y_gpu,
                    )

                else:
                    raise ValueError(f"Unknown alg: {alg}")

            # 4. Tính train–train distance (dùng đối xứng để tiết kiệm)
            for i in tqdm(range(train_len), desc=f"Train-train ({alg}) [run {run_idx+1}]"):
                X_computed[i, i] = 0.0
                for j in range(i + 1, train_len):
                    d = _pair_distance(X_train[i], X_train[j])
                    X_computed[i, j] = d
                    X_computed[j, i] = d
    
            # 5. Tính test–train distance
            for i in tqdm(range(test_len), desc=f"Test-train ({alg}) [run {run_idx+1}]"):
                for j in range(train_len):
                    X_test_computed[i, j] = _pair_distance(X_test[i], X_train[j])

        # 6. Chạy kNN với nhiều k, lấy best accuracy
        k_list = sorted(set(num_neighbor_list))
        accuracies = {}
        best_acc = np.nan
        best_k = None

        for k in k_list:
            if k > train_len:
                print(f"Skip k={k} (n_train={train_len} < k)")
                continue

            clf = neighbors.KNeighborsClassifier(n_neighbors=k, metric="precomputed")
            clf.fit(X_computed, y_train)
            y_pred = clf.predict(X_test_computed)
            acc = 100.0 * accuracy_score(y_test, y_pred)
            accuracies[k] = acc
            print(f"[Run {run_idx+1}] Accuracy of {k}NN: {acc:.2f} %")

            if (best_k is None) or (acc > best_acc):
                best_acc = acc
                best_k = k

        if best_k is None:
            best_acc = np.nan

        print(f"[Run {run_idx+1}] Best accuracy: {best_acc:.2f} % (k={best_k})")

        # 7. Tính MAP theo định nghĩa trong paper
        #    Nếu muốn fix k=1 như paper, đổi k_map = 1.
        k_map = best_k if best_k is not None else 1
        map_score = compute_map_knn_precomputed(
            X_computed, X_test_computed, y_train, y_test, k=k_map
        )
        print(f"[Run {run_idx+1}] Mean Average Precision (MAP) with k={k_map}: {map_score:.2f} %")

        # 8. Thời gian chạy lần này
        runtime_s = time.time() - t0
        print(f"[Run {run_idx+1}] Runtime: {runtime_s:.2f} s")

        # Lưu lại
        acc_list.append(best_acc)
        map_list.append(map_score)
        time_list.append(runtime_s)

    # ====== Sau num_runs lần, tính mean và variance ======
    acc_mean = float(np.mean(acc_list))
    acc_var = float(np.var(acc_list))      # nếu muốn sample variance: np.var(..., ddof=1)
    map_mean = float(np.mean(map_list))
    map_var = float(np.var(map_list))
    time_mean = float(np.mean(time_list))
    time_var = float(np.var(time_list))

    print("\n========== Summary over runs ==========")
    print(f"Accuracy: mean={acc_mean:.2f} %, var={acc_var:.4f}")
    print(f"MAP     : mean={map_mean:.2f} %, var={map_var:.4f}")
    print(f"Runtime : mean={time_mean:.2f} s, var={time_var:.4f}")

    # 9. Ghi ra Excel: dataset, accuracy_mean, accuracy_var, map_mean, map_var, runtime_mean, runtime_var
    dataset_key = f"{datapath}_{datatype}"
    out_file = f"{alg}.xlsx"
    cols = [
        "dataset",
        "accuracy_mean", "accuracy_var",
        "map_mean", "map_var",
        "runtime_mean", "runtime_var",
    ]
    
    new_row = {
        "dataset": dataset_key,
        "accuracy_mean": acc_mean,
        "accuracy_var": acc_var,
        "map_mean": map_mean,
        "map_var": map_var,
        "runtime_mean": time_mean,
        "runtime_var": time_var,
    }

    if os.path.exists(out_file):
        try:
            df = pd.read_excel(out_file, engine="openpyxl")
        except Exception:
            df = pd.read_excel(out_file)

        if "dataset" not in df.columns:
            df.insert(0, "dataset", "")

        mask = (df["dataset"] == dataset_key)
        if mask.any():
            for c in cols:
                df.loc[mask, c] = new_row[c]
        else:
            df = pd.concat(
                [df, pd.DataFrame([new_row], columns=cols)],
                ignore_index=True
            )
    else:
        df = pd.DataFrame([new_row], columns=cols)

    df = df[cols]
    df.to_excel(out_file, index=False, engine="openpyxl")

    # Trả về cho code bên ngoài dùng nếu cần
    return {
        "accuracy_mean": acc_mean,
        "accuracy_var": acc_var,
        "map_mean": map_mean,
        "map_var": map_var,
        "runtime_mean": time_mean,
        "runtime_var": time_var,
        "acc_runs": acc_list,
        "map_runs": map_list,
        "time_runs": time_list,
    }


In [ ]:
'''
sdatasets = [
        "ArrowHead",              # AH
        "BasicMotions",           # BM
        "BeetleFly",              # BF
        "CBF",                    # CBF
        "Chinatown",              # CT
        "CinCECGTorso",           # CET
        "DiatomSizeReduction",    # DSR
        "GunPointAgeSpan",        # GPA
        "GunPointMaleVersusFemale", # GPM
        "GunPointOldVersusYoung", # GPO
        "Ham",                    # Ham
        "InsectEPGRegularTrain",  # IERT
        "ItalyPowerDemand",       # IPD
        "Meat",                   # Meat
        "MelbournePedestrian",    # MP
        "MixedShapesSmallTrain",  # MS2T
        "MoteStrain",             # MS
        "OliveOil",               # O2
        "Plane",                  # Plane
        "SmoothSubspace",         # S2
    ]

datasets = [              # CT
        "MSRAction3D",
        "Weizmann",
        "SpokenArabicDigit"# MP
    ]
alg = "OPW"          # thuật toán muốn chạy
datatype = "Human_Actions" # UCR_TSL để dùng tslearn
out_file = f"{alg}_knn.xlsx"

num_runs = 1
if alg == "OTSW":
    num_runs = 5


for dataset in datasets:
    run_knn(dataset, datatype, alg, num_runs = num_runs)
'''

In [ ]:

# ============================================================
# ABLATION STUDY FOR OTSW PARAMETERS (k-NN Accuracy & MAP)
# ============================================================
"""
Ablation Study for OTSW Hyperparameters
=======================================

This section performs ablation study on OTSW hyperparameters measuring k-NN performance:
- Lambda (lam_time): (0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100)
- Max Depth: (5, 10, 15, 20, 25, 30)
- Number of Trees: (1, 3, 5, 7, 9, 11, 13, 15)
- Number of Clusters (k_split): (2, 4, 8, 16, 32)

Default values: lambda=5, depth=30, trees=5, num_cluster=2

When varying one parameter, all others are fixed at default values.
Results include Accuracy, MAP, and execution time for each configuration.
"""

import matplotlib.pyplot as plt

# ---------------------- Configuration ----------------------
# Default parameter values
ABLATION_DEFAULT_LAMBDA = 5
ABLATION_DEFAULT_DEPTH = 30
ABLATION_DEFAULT_TREES = 5
ABLATION_DEFAULT_NUM_CLUSTER = 2  # k_split

# Parameter ranges for ablation
ABLATION_LAMBDA_VALUES = [v**2 for v in [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100, 1000, 10000]]
# [1e-06, 2.5e-05, 0.0001, 0.0025, 0.01, 0.25, 1, 25, 100, 2500, 10000, 1000000, 100000000]
ABLATION_DEPTH_VALUES = [5, 10, 15, 20, 25, 30]
ABLATION_TREES_VALUES = [1, 3, 5, 7, 9, 11, 13, 15]
ABLATION_NUM_CLUSTER_VALUES = [2, 4, 8, 12, 16, 32, 64, 128, 256, 1024, 4096, 10000] 

# Dataset configuration
ABLATION_DATASET = "BasicMotions"
ABLATION_DATATYPE = "UCR_TSL"
ABLATION_LEAF_SIZE = 16
ABLATION_BASE_SEED = 0
ABLATION_K_NN = 1  # k for k-NN


def run_knn_ablation_single(
    X_train, y_train, X_test, y_test,
    lam_time=ABLATION_DEFAULT_LAMBDA,
    max_depth=ABLATION_DEFAULT_DEPTH,
    num_trees=ABLATION_DEFAULT_TREES,
    k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    leaf_size=ABLATION_LEAF_SIZE,
    seed=ABLATION_BASE_SEED,
    k_nn=ABLATION_K_NN,
):
    """
    Run k-NN with OTSW for a single parameter configuration.
    Returns: (Accuracy, MAP, time_total)
    """
    start_time = time.time()
    
    train_len = len(y_train)
    test_len = len(y_test)
    
    # Build sequences: test first, then train
    sequences = [np.asarray(X_test[i], dtype=float) for i in range(test_len)] + \
                [np.asarray(X_train[j], dtype=float) for j in range(train_len)]
    
    # Build OTSW model with multiple trees and average
    dist_accumulator = np.zeros((test_len, train_len), dtype=float)
    train_dist_accumulator = np.zeros((train_len, train_len), dtype=float)
    
    for t in range(num_trees):
        current_seed = seed + t
        
        model_otsw = build_otsw_tamle(
            sequences,
            lam_time=lam_time,
            leaf_size=leaf_size,
            max_depth=max_depth,
            seed=current_seed,
            k_split=k_split,
        )
        
        M_edge_mass = model_otsw.M  # (E, m_total)
        w = model_otsw.w.reshape(-1, 1)  # (E, 1)
        
        # Test-train distances
        for i in range(test_len):
            dist_all = (w * np.abs(M_edge_mass[:, i:i+1] - M_edge_mass)).sum(axis=0)
            dist_accumulator[i, :] += dist_all[test_len : test_len + train_len]
        
        # Train-train distances
        for i in range(train_len):
            idx_i = test_len + i
            dist_all = (w * np.abs(M_edge_mass[:, idx_i:idx_i+1] - M_edge_mass)).sum(axis=0)
            train_dist_accumulator[i, :] += dist_all[test_len : test_len + train_len]
    
    X_test_computed = dist_accumulator / num_trees
    X_computed = train_dist_accumulator / num_trees
    
    # Run k-NN
    clf = neighbors.KNeighborsClassifier(n_neighbors=k_nn, metric="precomputed")
    clf.fit(X_computed, y_train)
    y_pred = clf.predict(X_test_computed)
    acc = 100.0 * accuracy_score(y_test, y_pred)
    
    # Compute MAP
    map_score = compute_map_knn_precomputed(X_computed, X_test_computed, y_train, y_test, k=k_nn)
    
    time_total = time.time() - start_time
    
    return acc, map_score, time_total


def run_knn_ablation_for_param(
    X_train, y_train, X_test, y_test,
    param_name, param_values, num_runs=5, **fixed_params
):
    """
    Run ablation study for a single parameter with multiple runs.
    Returns: DataFrame with columns [param_value, Accuracy_mean, Accuracy_std, MAP_mean, MAP_std, Time_mean, Time_std]
    """
    results = []
    
    print(f"\n{'='*60}")
    print(f"Ablation Study (k-NN): {param_name}")
    print(f"Testing {len(param_values)} values: {param_values}")
    print(f"Number of runs per value: {num_runs}")
    print(f"Fixed params: {fixed_params}")
    print(f"{'='*60}")
    
    for val in param_values:
        params = fixed_params.copy()
        params[param_name] = val
        
        print(f"  Testing {param_name}={val}...")
        
        acc_runs = []
        map_runs = []
        time_runs = []
        
        for run_idx in range(num_runs):
            try:
                # Use different seed for each run
                params_with_seed = params.copy()
                params_with_seed['seed'] = ABLATION_BASE_SEED + run_idx * 100
                
                acc, map_score, time_total = run_knn_ablation_single(
                    X_train, y_train, X_test, y_test, **params_with_seed
                )
                acc_runs.append(acc)
                map_runs.append(map_score)
                time_runs.append(time_total)
                print(f"    Run {run_idx+1}/{num_runs}: ACC={acc:.2f}%, MAP={map_score:.2f}%, Time={time_total:.2f}s")
            except Exception as e:
                print(f"    Run {run_idx+1}/{num_runs}: ERROR: {e}")
                acc_runs.append(np.nan)
                map_runs.append(np.nan)
                time_runs.append(np.nan)
        
        # Calculate mean and std
        acc_mean = float(np.nanmean(acc_runs))
        acc_std = float(np.nanstd(acc_runs))
        map_mean = float(np.nanmean(map_runs))
        map_std = float(np.nanstd(map_runs))
        time_mean = float(np.nanmean(time_runs))
        time_std = float(np.nanstd(time_runs))
        
        print(f"    => Mean: ACC={acc_mean:.2f}±{acc_std:.2f}%, MAP={map_mean:.2f}±{map_std:.2f}%, Time={time_mean:.2f}±{time_std:.2f}s")
        
        results.append({
            param_name: val,
            "Accuracy_mean": acc_mean,
            "Accuracy_std": acc_std,
            "MAP_mean": map_mean,
            "MAP_std": map_std,
            "Time_mean": time_mean,
            "Time_std": time_std,
        })
    
    return pd.DataFrame(results)


def plot_knn_ablation_results(df, param_name, save_dir="."):
    """
    Plot ablation results: Accuracy, MAP, and Time vs parameter value with error bars (std).
    Saves directly to the specified directory (default: current directory).
    For lam_time, tick labels show sqrt(value) since the code applies sqrt internally.
    """
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    
    # For lam_time: display sqrt(value) as label (code uses sqrt(lam_time) internally)
    if param_name == "lam_time":
        x = [f"{v**0.5:.4g}" for v in df[param_name]]
        display_name = "λ"
    else:
        x = df[param_name].astype(str).tolist()
        display_name = param_name
    x_numeric = range(len(x))
    
    # Plot Accuracy with error bars
    axes[0].errorbar(x_numeric, df["Accuracy_mean"], yerr=df["Accuracy_std"], 
                     marker='o', linewidth=2, markersize=8, color='blue', 
                     capsize=4, capthick=1.5, elinewidth=1.5)
    axes[0].set_xlabel(display_name, fontsize=12)
    axes[0].set_ylabel("Accuracy (%)", fontsize=12)
    axes[0].set_title("Accuracy", fontsize=14)
    axes[0].set_xticks(x_numeric)
    axes[0].set_xticklabels(x, rotation=45, ha='right')
    axes[0].grid(True, alpha=0.3)
    
    # Plot MAP with error bars
    axes[1].errorbar(x_numeric, df["MAP_mean"], yerr=df["MAP_std"], 
                     marker='s', linewidth=2, markersize=8, color='green',
                     capsize=4, capthick=1.5, elinewidth=1.5)
    axes[1].set_xlabel(display_name, fontsize=12)
    axes[1].set_ylabel("MAP (%)", fontsize=12)
    axes[1].set_title("MAP", fontsize=14)
    axes[1].set_xticks(x_numeric)
    axes[1].set_xticklabels(x, rotation=45, ha='right')
    axes[1].grid(True, alpha=0.3)
    
    # Plot Time with error bars
    axes[2].errorbar(x_numeric, df["Time_mean"], yerr=df["Time_std"], 
                     marker='^', linewidth=2, markersize=8, color='red',
                     capsize=4, capthick=1.5, elinewidth=1.5)
    axes[2].set_xlabel(display_name, fontsize=12)
    axes[2].set_ylabel("Time (seconds)", fontsize=12)
    axes[2].set_title("Execution Time", fontsize=14)
    axes[2].set_xticks(x_numeric)
    axes[2].set_xticklabels(x, rotation=45, ha='right')
    axes[2].grid(True, alpha=0.3)
    
    plt.tight_layout()
    
    # Save figure directly in save_dir
    fig_path = os.path.join(save_dir, f"ablation_knn_{param_name}.png")
    plt.savefig(fig_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    
    print(f"✅ Plot saved to {fig_path}")
    return fig_path


# Number of runs for ablation study
ABLATION_NUM_RUNS = 5


def run_full_knn_ablation_study(
    dataset_name=ABLATION_DATASET,
    datatype=ABLATION_DATATYPE,
    save_dir=".",
    num_runs=ABLATION_NUM_RUNS,
):
    """
    Run complete ablation study for all OTSW parameters measuring k-NN performance.
    Each parameter configuration is run num_runs times (default: 5) to compute mean and std.
    Results are saved directly in save_dir (default: current directory).
    """
    print(f"\n{'#'*70}")
    print(f"# OTSW ABLATION STUDY (k-NN) ON DATASET: {dataset_name}")
    print(f"{'#'*70}")
    
    # Load dataset
    if datatype == "UCR_TSL":
        X_train, y_train, X_test, y_test = load_ucr_dataset_tsl("../data/UCR", dataset_name)
    elif datatype == "Human_Actions":
        X_train, y_train, X_test, y_test = load_human_action_dataset("../data/Human_Actions", dataset_name)
    else:
        raise ValueError(f"Unknown datatype: {datatype}")
    
    print(f"\nDataset: {dataset_name}")
    print(f"Train samples: {len(y_train)}, Test samples: {len(y_test)}")
    print(f"\nDefault parameters:")
    print(f"  - Lambda (lam_time): {ABLATION_DEFAULT_LAMBDA}")
    print(f"  - Max Depth: {ABLATION_DEFAULT_DEPTH}")
    print(f"  - Number of Trees: {ABLATION_DEFAULT_TREES}")
    print(f"  - Number of Clusters (k_split): {ABLATION_DEFAULT_NUM_CLUSTER}")
    print(f"  - Number of runs per config: {num_runs}")
    
    all_results = {}
    
    '''
    # 1. Ablation on Lambda (lam_time)
    print("\n" + "="*70)
    print("1. ABLATION ON LAMBDA (lam_time)")
    print("="*70)
    df_lambda = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="lam_time",
        param_values=ABLATION_LAMBDA_VALUES,
        num_runs=num_runs,
        max_depth=ABLATION_DEFAULT_DEPTH,
        num_trees=ABLATION_DEFAULT_TREES,
        k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    )
    df_lambda.to_csv(os.path.join(save_dir, "ablation_knn_lambda.csv"), index=False)
    plot_knn_ablation_results(df_lambda, "lam_time", save_dir)
    all_results["lambda"] = df_lambda
    '''
    # 2. Ablation on Max Depth
    print("\n" + "="*70)
    print("2. ABLATION ON MAX DEPTH")
    print("="*70)
    df_depth = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="max_depth",
        param_values=ABLATION_DEPTH_VALUES,
        num_runs=num_runs,
        lam_time=ABLATION_DEFAULT_LAMBDA,
        num_trees=ABLATION_DEFAULT_TREES,
        k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    )
    df_depth.to_csv(os.path.join(save_dir, "ablation_knn_depth.csv"), index=False)
    plot_knn_ablation_results(df_depth, "max_depth", save_dir)
    all_results["depth"] = df_depth
    
    # 3. Ablation on Number of Trees
    print("\n" + "="*70)
    print("3. ABLATION ON NUMBER OF TREES")
    print("="*70)
    df_trees = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="num_trees",
        param_values=ABLATION_TREES_VALUES,
        num_runs=num_runs,
        lam_time=ABLATION_DEFAULT_LAMBDA,
        max_depth=ABLATION_DEFAULT_DEPTH,
        k_split=ABLATION_DEFAULT_NUM_CLUSTER,
    )
    df_trees.to_csv(os.path.join(save_dir, "ablation_knn_trees.csv"), index=False)
    plot_knn_ablation_results(df_trees, "num_trees", save_dir)
    all_results["trees"] = df_trees
    '''
    # 4. Ablation on Number of Clusters (k_split)
    print("\n" + "="*70)
    print("4. ABLATION ON NUMBER OF CLUSTERS (k_split)")
    print("="*70)
    df_cluster = run_knn_ablation_for_param(
        X_train, y_train, X_test, y_test,
        param_name="k_split",
        param_values=ABLATION_NUM_CLUSTER_VALUES,
        num_runs=num_runs,
        lam_time=ABLATION_DEFAULT_LAMBDA,
        max_depth=ABLATION_DEFAULT_DEPTH,
        num_trees=ABLATION_DEFAULT_TREES,
    )
    df_cluster.to_csv(os.path.join(save_dir, "ablation_knn_cluster.csv"), index=False)
    plot_knn_ablation_results(df_cluster, "k_split", save_dir)
    all_results["cluster"] = df_cluster
    '''
    # Summary
    print("\n" + "#"*70)
    print("# ABLATION STUDY (k-NN) COMPLETE")
    print("#"*70)
    print(f"\nResults saved to {save_dir}:")
    print("  - ablation_knn_lambda.csv + ablation_knn_lam_time.png")
    print("  - ablation_knn_depth.csv + ablation_knn_max_depth.png")
    print("  - ablation_knn_trees.csv + ablation_knn_num_trees.png")
    print("  - ablation_knn_cluster.csv + ablation_knn_k_split.png")
    
    return all_results


# ---------------------- Usage ----------------------
# Run the full ablation study (5 runs per config by default):
all_results = run_full_knn_ablation_study(dataset_name="ItalyPowerDemand", datatype="UCR_TSL", save_dir=".", num_runs=1)

#
# Or run individual parameter ablations:
#   X_train, y_train, X_test, y_test = load_ucr_dataset_tsl("../data/UCR", "BasicMotions")
#   df_lambda = run_knn_ablation_for_param(X_train, y_train, X_test, y_test, "lam_time", ABLATION_LAMBDA_VALUES, num_runs=5, max_depth=30, num_trees=5, k_split=2)
#
# CSV output columns: param_value, Accuracy_mean, Accuracy_std, MAP_mean, MAP_std, Time_mean, Time_std
